In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
from xarray.coding.times import CFDatetimeCoder
from pathlib import Path


In [ ]:

def MAKE_XGMOD(output_path="/ec/vol/centaur/pajm/training_data.parquet"):
    """
    Build an XGBoost-compatible training dataset by combining
    climate, vegetation, and anthropogenic data from multiple NetCDF sources.
    """

    # -----------------------------
    # CONFIG & CONSTANTS
    # -----------------------------
    base_path = Path("/ec/vol/centaur/pajm/DATA")
    years = list(range(2003, 2004))          # extend this list if needed
    months = list(range(1, 12))            # or range(1, 13) for all months
    sample_frac = 1 / 100   # keep ~10% of total samples

    # -----------------------------
    # LOAD STATIC DATASETS
    # -----------------------------
    time_coder = CFDatetimeCoder(use_cftime=True)

    print("Loading static datasets...")
    PO = xr.open_dataset(base_path / "CLIMATE/POP_2020.nc")
    UR = xr.open_dataset(base_path / "CLIMATE/urban_C.nc", decode_times=time_coder)
    RD = xr.open_dataset(base_path / "CLIMATE/road_density_2015_c.nc")

    ur = UR.vegdiff.squeeze().values.flatten()
    po = PO.population_density.values.flatten()
    rd = RD.road_length.values.flatten()

    all_samples = []

    # -----------------------------
    # MAIN LOOP
    # -----------------------------
    for year in years:
        print(f"Processing year {year}...")

        for month in months:
            mon = f"{month:02d}"  # zero-padded month

            # Define paths compactly
            ds_paths = {
                "AF": f"ACTIVE_FIRE_MAP_{year}_{mon}.nc",
                "FU": f"TV_CFUEL_MAP_{year}_{mon}.nc",
                "DF": f"TV_DFMC_MAP_{year}_{mon}.nc",
                "LF": f"TV_LFMC_MAP_{year}_{mon}.nc",
                "PR": f"DFMC_DATA/P/P_{year}_{mon}_C.nc",
                "T2": f"DFMC_DATA/T2M/T2M_{year}_{mon}_C.nc",
                "D2": f"DFMC_DATA/D2M/D2M_{year}_{mon}_C.nc",
                "WS": f"DFMC_DATA/10W/WS_{year}_{mon}_C.nc",
            }

            # Load datasets efficiently with context managers
            with xr.open_dataset(base_path / ds_paths["AF"]) as AF, \
                 xr.open_dataset(base_path / ds_paths["FU"]) as FU, \
                 xr.open_dataset(base_path / ds_paths["DF"]) as DF, \
                 xr.open_dataset(base_path / ds_paths["LF"]) as LF, \
                 xr.open_dataset(base_path / ds_paths["PR"]) as PR, \
                 xr.open_dataset(base_path / ds_paths["T2"]) as T2, \
                 xr.open_dataset(base_path / ds_paths["D2"]) as D2, \
                 xr.open_dataset(base_path / ds_paths["WS"]) as WS:

                days = len(AF.ACTIVE_FIRE)

                # Process only the first day (change np.arange(1) to range(len(AF.ACTIVE_FIRE)) if needed)
                for i in np.arange(days):
                    af = AF.ACTIVE_FIRE[i].values.flatten()
                    fu_ll = FU.Live_Leaf[i].values.flatten()
                    fu_lw = FU.Live_Wood[i].values.flatten()
                    fu_df = FU.Dead_Foliage[i].values.flatten()
                    fu_dw = FU.Dead_Wood[i].values.flatten()
                    df = DF.DFMC_Foliage[i].values.flatten()
                    dw = DF.DFMC_Wood[i].values.flatten()
                    lf = LF.LFMC[i].values.flatten()
                    pr = PR.var228[i].values.flatten()
                    t2 = T2.var167[i].values.flatten()
                    d2 = D2.var168[i].values.flatten()
                    ws = WS.ws[i].values.flatten()

                    # Mask where total fuel > 0
                    ft = fu_ll + fu_lw + fu_df + fu_dw
                    mask = ft > 0.0

                    # Build dataframe for valid pixels
                    dfx = pd.DataFrame({
                        "AF": af[mask],
                        "PR": pr[mask],
                        "T2": t2[mask],
                        "D2": d2[mask],
                        "WS": ws[mask],
                        "FU_LL": fu_ll[mask],
                        "FU_LW": fu_lw[mask],
                        "FU_DF": fu_df[mask],
                        "FU_DW": fu_dw[mask],
                        "DF": df[mask],
                        "DW": dw[mask],
                        "LF": lf[mask],
                        "UR": ur[mask],
                        "PO": po[mask],
                        "RD": rd[mask],
                    }, dtype=float)

                    dfx.dropna(inplace=True)

                    # Random sample for manageable dataset
                    if len(dfx) > 0:
                        n_samples = max(1, int(len(dfx) * sample_frac))
                        dfx = dfx.sample(n=n_samples, random_state=1)
                        all_samples.append(dfx)

    # -----------------------------
    # COMBINE & SAVE
    # -----------------------------
    if not all_samples:
        print("⚠️ No samples generated. Check data masks or paths.")
        return

    print("Combining sampled data...")
    dfa = pd.concat(all_samples, ignore_index=True)

    print(f"Saving dataset → {output_path}")
    dfa.to_parquet(output_path)

    print("✅ Training dataset ready.")
    return dfa



In [ ]:

MAKE_XGMOD()